# DS2002 · Normalize and Join Nested JSON

**Lab — 2026-10-02 · Fall 2026**  

---

## Lab 06 — Normalize and Join Nested JSON

Two JSON payloads of the kind an API hands you. Flatten each, join them, and answer the questions.

Before you write anything, answer this for yourself: **what is one row of the table you are building?** Every argument you pass to `json_normalize` follows from that answer, and getting it wrong is the reason this lab takes people two hours instead of forty minutes.

One vendor in the orders is not in the vendor list. That is on purpose.

In [2]:
orders_json = [
  {'order':1,'vendor_id':'V-01','lines':[{'item':'Cheeseburger','qty':2},{'item':'Soda','qty':1}]},
  {'order':2,'vendor_id':'V-10','lines':[{'item':'Foam Finger','qty':1}]},
  {'order':3,'vendor_id':'V-07','lines':[{'item':'Hot Dog','qty':3}]},
]
vendors_json = [
  {'vendor_id':'V-01','name':'Hoos Burgers','zone':'A'},
  {'vendor_id':'V-10','name':'Cav Merch','zone':'A'},
]

**One row of my result is one:** One row of my result is one item purchased within an order, linked to its vendor information.

### Q1 — Flatten the orders to one row per line item.

Keep `order` and `vendor_id` on every row.

*Expected: 4 rows, and the quantities should sum to 7.*

In [3]:
import pandas as pd

orders_df = pd.json_normalize(
    orders_json,
    record_path='lines',
    meta=['order', 'vendor_id']
)

print(orders_df)

assert len(orders_df) == 4
assert orders_df['qty'].sum() == 7

print("Row count:", len(orders_df))
print("Total quantity:", orders_df['qty'].sum())

           item  qty order vendor_id
0  Cheeseburger    2     1      V-01
1          Soda    1     1      V-01
2   Foam Finger    1     2      V-10
3       Hot Dog    3     3      V-07
Row count: 4
Total quantity: 7


### Q2 — Join the vendor information on.

Left join, `indicator=True`. Report which vendor did not match and how many units are attached to it.

In [4]:
vendors_df = pd.json_normalize(vendors_json)

joined_df = orders_df.merge(
    vendors_df,
    on='vendor_id',
    how='left',
    indicator=True,
    validate='many_to_one'
)

print(joined_df)

unmatched = joined_df[joined_df['_merge'] == 'left_only']

print("Unmatched vendor:", unmatched['vendor_id'].unique())
print("Units attached to unmatched vendor:", unmatched['qty'].sum())

           item  qty order vendor_id          name zone     _merge
0  Cheeseburger    2     1      V-01  Hoos Burgers    A       both
1          Soda    1     1      V-01  Hoos Burgers    A       both
2   Foam Finger    1     2      V-10     Cav Merch    A       both
3       Hot Dog    3     3      V-07           NaN  NaN  left_only
Unmatched vendor: ['V-07']
Units attached to unmatched vendor: 3


### Q3 — Total quantity per vendor name.

Decide how the unmatched vendor appears in this table, and say why in a comment.

In [5]:
joined_df['name'] = joined_df['name'].fillna('Unknown Vendor')

qty_by_vendor = (
    joined_df.groupby('name')['qty']
    .sum()
    .sort_values(ascending=False)
)

print(qty_by_vendor)

# Check that all 7 units are accounted for
assert qty_by_vendor.sum() == 7

name
Hoos Burgers      3
Unknown Vendor    3
Cav Merch         1
Name: qty, dtype: int64


### Q4 — Which zone sold the most items?

Careful: if the unmatched vendor has no zone, your zone totals will not add up to 7. State what your total is and where the difference went.

In [6]:
zone_totals = (
    joined_df.groupby('zone')['qty']
    .sum()
    .sort_values(ascending=False)
)

print(zone_totals)

print("Zone total:", zone_totals.sum())
print("Missing units:", joined_df['qty'].sum() - zone_totals.sum())
print("Zone with most items:", zone_totals.idxmax())

zone
A    4
Name: qty, dtype: int64
Zone total: 4
Missing units: 3
Zone with most items: A


Zone A sold the most items with 4 units. The zone total is 4 instead of 7 because vendor V-07 has no matching vendor information or zone.
Pandas groupby excludes missing zone values by default, so the remaining 3 units are not included in the zone totals.

### Q5 — Handle a ragged payload.

A second batch arrives and it is not uniform: one order has no `lines` key at all, and one line item is missing its `qty`.

**TODO:** flatten what you can without crashing, and produce a frame where a missing quantity is distinguishable from a quantity of zero. Do **not** use `fillna(0)`.

In [7]:
ragged_json = [
    {'order': 4, 'vendor_id': 'V-01', 'lines': [{'item': 'Soda', 'qty': 2}]},
    {'order': 5, 'vendor_id': 'V-10'},                       # no lines at all
    {'order': 6, 'vendor_id': 'V-01', 'lines': [{'item': 'Fries'}]},  # no qty
]

for order in ragged_json:
    order.setdefault('lines', [])

ragged_df = pd.json_normalize(
    ragged_json,
    record_path='lines',
    meta=['order', 'vendor_id']
)

ragged_df['qty'] = pd.to_numeric(
    ragged_df['qty'],
    errors='coerce'
)

print(ragged_df)

print("Total rows:", len(ragged_df))
print("Missing quantities:", ragged_df['qty'].isna().sum())

    item  qty order vendor_id
0   Soda  2.0     4      V-01
1  Fries  NaN     6      V-01
Total rows: 2
Missing quantities: 1


### Q6 — Validate

In [8]:
assert len(orders_df) == 4
assert orders_df['qty'].sum() == 7
assert len(joined_df) == len(orders_df), 'the vendor join changed the row count'

print('checks passed.')

checks passed.


### Write-up

**a)** Why is "one row per line item" the useful shape here? What question becomes easy that would be hard with one row per order?

**b)** In Q5, what is the practical difference between a missing quantity and a quantity of zero, and what breaks downstream if you conflate them?

a) One row per line item makes it easier to analyze individual products and quantities sold. For example, I can calculate the total number of items sold by each vendor or zone without having to work through nested lists inside each order. If I kept one row per order, these calculations would be more complicated because an order can contain multiple items.

b) A missing quantity means the data was not provided, while a quantity of zero means no units were ordered. In Q5, the Fries item has a missing quantity, so I kept it as NaN instead of replacing it with zero. If I treated missing values as zero, I would incorrectly assume that no Fries were sold, which could make sales totals and averages inaccurate and hide data quality issues.